# Conv2d

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mitchell-Mirano/sorix/blob/develop/docs/learn/layers/08-Conv2d.ipynb)
[![Open in GitHub](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/Mitchell-Mirano/sorix/blob/develop/docs/learn/layers/08-Conv2d.ipynb)
[![Open in Docs](https://img.shields.io/badge/Open%20in-Docs-blue?logo=readthedocs)](http://127.0.0.1:8000/sorix/learn/layers/08-Conv2d)

The **Conv2d** layer applies a 2D spatial convolution over an input signal composed of several input planes. It acts as a local feature extractor through cross-correlation filters, capturing hierarchical patterns such as edges and textures.

## Mathematical definition

Let $\mathbf{X} \in \mathbb{R}^{N \times C_{in} \times H \times W}$ be the input tensor and the learning parameters be defined by an array of kernels $\mathbf{W} \in \mathbb{R}^{C_{out} \times C_{in} \times K_H \times K_W}$, alongside an optional bias vector $\mathbf{b} \in \mathbb{R}^{C_{out}}$.

### Forward Computation

The cross-correlation logic slides the filters across spatial dimensions according to stride factors ($s_h, s_w$) and zero constraints (padding applied as $p_h, p_w$). The output element $Y_{n, c_{out}, h_{out}, w_{out}}$ evaluates to:

$$
\mathbf{Y}_{n, c_{out}, h_{out}, w_{out}} = b_{c_{out}} + \sum_{c_{in}=0}^{C_{in}-1} \sum_{k_h=0}^{K_H-1} \sum_{k_w=0}^{K_W-1} \mathbf{W}_{c_{out}, c_{in}, k_h, k_w} \cdot \mathbf{X}_{n, c_{in}, h_{out} s_h + k_h - p_h, w_{out} s_w + k_w - p_w}
$$

The resulting dimensions of the output tensor $\mathbf{Y} \in \mathbb{R}^{N \times C_{out} \times H_{out} \times W_{out}}$ are exactly quantified by:

$$
H_{out} = \left\lfloor \frac{H + 2p_h - K_H}{s_h} + 1 \right\rfloor \quad \text{and} \quad W_{out} = \left\lfloor \frac{W + 2p_w - K_W}{s_w} + 1 \right\rfloor
$$

## Parameterization and Gradients (Backpropagation)

The power of a Convolution lies in maintaining differentiability locally. Let $\mathcal{L}$ be the overarching scalar loss function, and let $\frac{\partial \mathcal{L}}{\partial \mathbf{Y}}$ be the upstream error gradient propagated from subsequent layers.

1. **Gradient w.r.t. Bias ($\mathbf{b}$):**
   The bias contributes uniformly across output dimensions. Thus, its gradient is the total channel-specific summation:
   $$
   \frac{\partial \mathcal{L}}{\partial b_{c_{out}}} = \sum_{n=0}^{N-1} \sum_{h_{out}=0}^{H_{out}-1} \sum_{w_{out}=0}^{W_{out}-1} \left( \frac{\partial \mathcal{L}}{\partial \mathbf{Y}} \right)_{n, c_{out}, h_{out}, w_{out}}
   $$

2. **Gradient w.r.t. Weights ($\mathbf{W}$):**
   Weight updates evaluate the correlation between upstream gradients and input patches where those weights were mapped.
   $$
   \frac{\partial \mathcal{L}}{\partial \mathbf{W}_{c_{out}, c_{in}, k_h, k_w}} = \sum_{n, h_{out}, w_{out}} \left( \frac{\partial \mathcal{L}}{\partial \mathbf{Y}} \right)_{n, c_{out}, h_{out}, w_{out}} \cdot \mathbf{X}_{n, c_{in}, h_{out} s_h + k_h - p_h, w_{out} s_w + k_w - p_w}
   $$

3. **Gradient w.r.t. Input ($\mathbf{X}$):**
   We map spatial errors back to inputs, managing overlap accumulations algebraically natively equivalent to performing a *transposed convolution* with appropriately inverted filters (often optimized computationally via overlapping `im2col` to `col2im` arrays).
   $$
   \frac{\partial \mathcal{L}}{\partial \mathbf{X}_{n, c_{in}, h_{in}, w_{in}}} = \sum_{c_{out}} \sum_{h_{out}, w_{out}} \sum_{k_h, k_w \in \Omega} \left( \frac{\partial \mathcal{L}}{\partial \mathbf{Y}} \right)_{n, c_{out}, h_{out}, w_{out}} \cdot \mathbf{W}_{c_{out}, c_{in}, k_h, k_w}
   $$
   *(where the subset $\Omega$ identifies all mapping instances dynamically overlapping $(h_{in}, w_{in})$ given the stride context).*


In [1]:
# Uncomment the next line and run this cell to install sorix
#!pip install 'sorix @ git+https://github.com/Mitchell-Mirano/sorix.git@develop'

In [2]:
from sorix import tensor
from sorix.nn import Conv2d
import numpy as np

In [3]:
# Create a random input standardized format: (Batch Size, Channels, Height, Width)
N, C_in, H, W = 2, 3, 32, 32
X = tensor(np.random.randn(N, C_in, H, W).astype(np.float32))

print("Input tensor shape (N, C, H, W):", X.shape)

Input tensor shape (N, C, H, W): sorix.Size([2, 3, 32, 32])


In [4]:
# Instantiate a Convolutional layer:
# Transforms 3 channels to 16 feature maps using a 3x3 kernel.
conv = Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1, stride=1)

print("Weights parameter shape:", conv.W.shape)
print("Bias parameter shape:", conv.b.shape)

Weights parameter shape: sorix.Size([16, 3, 3, 3])
Bias parameter shape: sorix.Size([16, 1])


In [5]:
# Forward pass
Y = conv(X)
print("Output dimension (padding preserves 32x32):", Y.shape)

Output dimension (padding preserves 32x32): sorix.Size([2, 16, 32, 32])
